# TASK 5.1: THỰC THI FULL-BENCHMARK TRÊN 18 TẬP TEST HAI CHIỀU
### ĐÁNH GIÁ TOÀN DIỆN MÔ HÌNH FATFORMER-XLA FINETUNE PHƯƠNG ÁN A (PLAN A - BLACKWELL GPU)

> **Người thực hiện**: Thành viên C *(Training & Evaluation Lead)*  
> **Mã nhiệm vụ**: `Task 5.1` *(Milestone 4 - Tuần 5)*  
> **Môi trường khuyến nghị**: Google Colab GPU Tesla T4 (Miễn phí / 1.5–2 CU/giờ) hoặc GPU L4.  
> **Mô hình kiểm thử**: Checkpoint chính thức `fatformer_plan_a_final.pth` (3.28 GB, kế thừa Epoch 3 kỷ lục: Val ACC 99.84%, AP 100%, AUC 100%).  
> **Mục tiêu**:  
> 1. **Chiều 1 (Cross-Generator Generalization)**: Kiểm thử trên 8 kiến trúc GANs và 10 kiến trúc Diffusion ở trạng thái Sạch (Clean). Kỳ vọng khôi phục Clean ACC $\ge 96.0\%$.  
> 2. **Chiều 2 (Degradation Robustness)**: Kiểm thử khả năng kháng suy biến trên toàn bộ 6 biến thể ($Q=30, 50, 70$, $\text{Blur}_1, \text{Blur}_2$, $\text{Down-Up}$). Kỳ vọng $Q=30$ ACC $\ge 65.0\%$, Fake ACC $\ge 50.0\%$.  
> 3. **Xuất Ma Trận Master**: Lưu `final_full_benchmark_matrix.csv` trực tiếp trên Google Drive 5TB `MyDrive/Fatformer/log/`.

---

### QUY TRÌNH THỰC THI 7 BƯỚC TỰ ĐỘNG:
1. **Bước 1**: Kết nối Google Drive 5TB & Đồng bộ mã nguồn Git nhánh `hoang` mới nhất.
2. **Bước 2**: Kiểm tra phần cứng GPU T4/L4 và xác thực Checkpoint Finetune Plan A (`fatformer_plan_a_final.pth`).
3. **Bước 3**: Trích xuất dữ liệu kiểm thử chống nghẽn I/O sang SSD NVMe máy ảo Colab.
4. **Bước 4**: Thực thi Chiều 1: Clean Benchmark trên 18 tập test (`tools/full_eval.py`).
5. **Bước 5**: Thực thi Chiều 2: Degraded Benchmark trên 6 biến thể vật lý (`tools/eval_degraded.py`).
6. **Bước 6**: Hợp nhất Ma Trận Hai Chiều Master Benchmark (`final_full_benchmark_matrix.csv`).
7. **Bước 7**: Trực quan hóa đối đầu toàn diện và xuất biểu đồ phục vụ Báo cáo & Slide bảo vệ.

In [ ]:
# ================================================================================
# BƯỚC 1: KẾT NỐI GOOGLE DRIVE 5TB & ĐỒNG BỘ MÃ NGUỒN GIT TỪ NHÁNH HOANG
# ================================================================================
from google.colab import drive
import os, sys

print("[*] Đang kết nối Google Drive 5TB...")
drive.mount('/content/drive')
DRIVE_ROOT = "/content/drive/MyDrive/Fatformer"
assert os.path.exists(DRIVE_ROOT), f"LỖI: Không tìm thấy thư mục {DRIVE_ROOT} trên Google Drive!"
print(f"[✓] Đã kết nối thành công kho lưu trữ: {DRIVE_ROOT}")

# Đồng bộ repository từ nhánh hoang
REPO_DIR = "/content/fatformer-xla"
if not os.path.exists(REPO_DIR):
    print("[*] Đang clone repository nhánh hoang...")
    !git clone -b hoang https://github.com/LeeVHoangtk3/fatformer-xla.git {REPO_DIR}
else:
    print("[*] Đang cập nhật mã nguồn mới nhất từ branch hoang...")
    %cd {REPO_DIR}
    !git checkout hoang
    !git pull origin hoang

%cd {REPO_DIR}
print(f"[✓] Thư mục làm việc hiện tại: {os.getcwd()}")

# Cài đặt thư viện phụ trợ
!pip install -q ftfy regex tqdm tabulate matplotlib seaborn pandas

In [ ]:
# ================================================================================
# BƯỚC 2: KIỂM TRA PHẦN CỨNG GPU & XÁC THỰC CÁC TỆP TRỌNG SỐ TRÊN DRIVE 5TB
# ================================================================================
import os, torch

assert torch.cuda.is_available(), 'LỖI: Vui lòng bật GPU trong Runtime -> Change runtime type -> T4/L4 GPU!'
gpu_name = torch.cuda.get_device_name(0)
gpu_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'[✓] GPU khả dụng: {gpu_name} ({gpu_vram:.2f} GB VRAM) - Sẵn sàng cho Full-Benchmark!')

# 1. Tự động tìm checkpoint Plan A tối ưu nhất trên Google Drive
candidate_ckpts = [
    f'{DRIVE_ROOT}/checkpoint/fatformer_plan_a_final.pth',
    f'{DRIVE_ROOT}/checkpoint/model_best.pth',
    f'{DRIVE_ROOT}/checkpoint/fatformer_plan_a_weights_only.pth',
    f'{DRIVE_ROOT}/checkpoint/fatformer_srm_robust_final.pth'
]
CKPT_FINAL = None
for c in candidate_ckpts:
    if os.path.exists(c):
        CKPT_FINAL = c
        break

assert CKPT_FINAL is not None, f'LỖI: Chưa tìm thấy file checkpoint trong {DRIVE_ROOT}/checkpoint/! Vui lòng tải file fatformer_plan_a_final.pth lên Drive.'

# 2. Định vị file CLIP ViT-L-14
CLIP_PATH = f'{DRIVE_ROOT}/pretrained/ViT-L-14.pt'
if not os.path.exists(CLIP_PATH):
    alt_clip = f'{DRIVE_ROOT}/ViT-L-14.pt'
    if os.path.exists(alt_clip):
        CLIP_PATH = alt_clip

print('\n--- KIỂM TOÁN TÍNH TOÀN VẸN CHECKPOINTS CHO TASK 5.1 ---')
print(f' • Checkpoint đánh giá:  {CKPT_FINAL} ({os.path.getsize(CKPT_FINAL)/(1024**2):.2f} MB)')
print(f' • Backbone CLIP ViT-L:   {CLIP_PATH} ({os.path.getsize(CLIP_PATH)/(1024**2):.2f} MB)')
print('[✓] Hệ thống sẵn sàng nạp mô hình vào pipeline kiểm thử 2 chiều!')


In [ ]:
# ================================================================================
# BƯỚC 3: GIẢI NÉN CHỐNG NGHẼN I/O 18 TẬP KIỂM THỬ SANG Ổ SSD NVME CỦA COLAB
# ================================================================================
import os, tarfile, time, shutil

LOCAL_CLEAN_DIR = "/content/dataset_local/test_clean"
LOCAL_DEG_DIR = "/content/dataset_local/test_degraded"
DEST_BASE = "/content/dataset_local"
os.makedirs(LOCAL_CLEAN_DIR, exist_ok=True)
os.makedirs(DEST_BASE, exist_ok=True)

# --------------------------------------------------------------------------------
# 1. GIẢI NÉN TẬP TEST CLEAN (GỒM 2 TỆP THỰC TẾ TRÊN DRIVE 5TB: GANS + DIFFUSION)
# --------------------------------------------------------------------------------
GANS_TAR = f"{DRIVE_ROOT}/datasets/test_benchmark_gans.tar"
DIFF_TAR = f"{DRIVE_ROOT}/datasets/test_benchmark_diffusion.tar"

subsets_clean = [d for d in os.listdir(LOCAL_CLEAN_DIR) if os.path.isdir(os.path.join(LOCAL_CLEAN_DIR, d))] if os.path.exists(LOCAL_CLEAN_DIR) else []
if len(subsets_clean) >= 18:
    print(f"[✓] Tập test Clean đã có sẵn tại SSD: {LOCAL_CLEAN_DIR} ({len(subsets_clean)}/18 subsets)!")
else:
    # 1.1 Giải nén 10 tập Diffusion (1.22 GB - Tốc độ đọc ~10-15s)
    if os.path.exists(DIFF_TAR):
        print(f"[*] Đang giải nén 10 tập Diffusion từ {DIFF_TAR} (1.22 GB)...")
        t0 = time.time()
        with tarfile.open(DIFF_TAR, "r") as tar:
            tar.extractall(LOCAL_CLEAN_DIR)
        # Tự động làm phẳng nếu file nén có thư mục con diffusion_test/
        diff_inner = os.path.join(LOCAL_CLEAN_DIR, "diffusion_test")
        if os.path.exists(diff_inner):
            for item in os.listdir(diff_inner):
                src = os.path.join(diff_inner, item)
                dst = os.path.join(LOCAL_CLEAN_DIR, item)
                if not os.path.exists(dst):
                    shutil.move(src, dst)
            shutil.rmtree(diff_inner, ignore_errors=True)
        print(f"  -> [✓] Hoàn tất 10 tập Diffusion trong {time.time() - t0:.1f}s!")
    else:
        print(f"[!] CẢNH BÁO: Không tìm thấy {DIFF_TAR} trên Drive!")

    # 1.2 Giải nén 8 tập GANs (18.74 GB - Tốc độ đọc ~2-3 phút)
    if os.path.exists(GANS_TAR):
        print(f"[*] Đang giải nén 8 tập GANs từ {GANS_TAR} (18.74 GB)...")
        t0 = time.time()
        with tarfile.open(GANS_TAR, "r") as tar:
            tar.extractall(LOCAL_CLEAN_DIR)
        # Tự động làm phẳng nếu file nén có thư mục con gans_test/
        gans_inner = os.path.join(LOCAL_CLEAN_DIR, "gans_test")
        if os.path.exists(gans_inner):
            for item in os.listdir(gans_inner):
                src = os.path.join(gans_inner, item)
                dst = os.path.join(LOCAL_CLEAN_DIR, item)
                if not os.path.exists(dst):
                    shutil.move(src, dst)
            shutil.rmtree(gans_inner, ignore_errors=True)
        print(f"  -> [✓] Hoàn tất 8 tập GANs trong {time.time() - t0:.1f}s!")
    else:
        print(f"[!] CẢNH BÁO: Không tìm thấy {GANS_TAR} trên Drive!")

    final_clean = [d for d in os.listdir(LOCAL_CLEAN_DIR) if os.path.isdir(os.path.join(LOCAL_CLEAN_DIR, d))]
    print(f"[✓] Tổng cộng dữ liệu Clean sẵn sàng: {len(final_clean)} subsets tại {LOCAL_CLEAN_DIR}!")

# --------------------------------------------------------------------------------
# 2. GIẢI NÉN TẬP TEST DEGRADED (19.55 GB - 6 BIẾN THỂ SUY BIẾN VẬT LÝ)
# --------------------------------------------------------------------------------
DEGRADED_TAR = f"{DRIVE_ROOT}/datasets/test_degraded.tar"

variants_deg = [d for d in os.listdir(LOCAL_DEG_DIR) if os.path.isdir(os.path.join(LOCAL_DEG_DIR, d))] if os.path.exists(LOCAL_DEG_DIR) else []
if len(variants_deg) >= 6:
    print(f"[✓] Tập test Degraded đã có sẵn tại SSD: {LOCAL_DEG_DIR} ({len(variants_deg)}/6 variants)!")
elif os.path.exists(DEGRADED_TAR):
    print(f"[*] Đang trích xuất 'test_degraded.tar' (19.55 GB) sang SSD NVMe Colab...")
    t0 = time.time()
    !tar -xf "{DEGRADED_TAR}" -C "{DEST_BASE}"
    variants_deg = [d for d in os.listdir(LOCAL_DEG_DIR) if os.path.isdir(os.path.join(LOCAL_DEG_DIR, d))] if os.path.exists(LOCAL_DEG_DIR) else []
    print(f"[✓] Giải nén Degraded hoàn tất trong {time.time() - t0:.1f}s: {len(variants_deg)} variants tại {LOCAL_DEG_DIR}!")
else:
    print(f"[!] CẢNH BÁO: Không tìm thấy {DEGRADED_TAR} trên Drive!")


In [ ]:
# ================================================================================
# BƯỚC 4: THỰC THI CHIỀU 1: FULL BENCHMARK CLEAN TRÊN 18 TẬP TEST (TOOLS/FULL_EVAL.PY)
# ================================================================================
OUTPUT_CLEAN_CSV = f"{DRIVE_ROOT}/log/task_5.1_clean_results.csv"
OUTPUT_CLEAN_MD = f"{DRIVE_ROOT}/log/task_5.1_clean_report.md"

cmd_clean = (
    f"python tools/full_eval.py "
    f"--checkpoint {CKPT_FINAL} "
    f"--clip_path {CLIP_PATH} "
    f"--test_root {LOCAL_CLEAN_DIR} "
    f"--num_vit_adapter 8 "
    f"--use_srm "
    f"--use_gating "
    f"--batch_size 32 "
    f"--num_workers 4 "
    f"--output_csv {OUTPUT_CLEAN_CSV} "
    f"--output_markdown {OUTPUT_CLEAN_MD}"
)

print("[*] Đang thực thi Full Benchmark trên 18 tập test SẠCH (Clean)...")
!{cmd_clean}


In [ ]:
# ================================================================================
# BƯỚC 5: THỰC THI CHIỀU 2: DEGRADED ROBUSTNESS TRÊN 6 BIẾN THỂ (TOOLS/EVAL_DEGRADED.PY)
# ================================================================================
OUTPUT_DEG_CSV = f"{DRIVE_ROOT}/log/task_5.1_degraded_results.csv"
OUTPUT_DEG_MD = f"{DRIVE_ROOT}/log/task_5.1_degraded_report.md"

# Chế độ kiểm thử suy biến:
# --variants all: Đo toàn bộ 6 biến thể (jpeg_q30, jpeg_q50, jpeg_q70, blur_s1, blur_s2, down_up)
# --samples_per_subset 500: Lấy mẫu đại diện khoa học (250 real + 250 fake) tương tự Task 2.3 để chạy tối ưu GPU
cmd_deg = (
    f"python tools/eval_degraded.py "
    f"--checkpoint {CKPT_FINAL} "
    f"--clip_path {CLIP_PATH} "
    f"--degraded_root {LOCAL_DEG_DIR} "
    f"--variants all "
    f"--num_vit_adapter 8 "
    f"--samples_per_subset 500 "
    f"--batch_size 32 "
    f"--num_workers 4 "
    f"--use_srm "
    f"--use_gating "
    f"--clean_csv {OUTPUT_CLEAN_CSV} "
    f"--output_csv {OUTPUT_DEG_CSV} "
    f"--output_markdown {OUTPUT_DEG_MD}"
)

print("[*] Đang thực thi Degraded Robustness Benchmark trên 6 biến thể suy biến...")
!{cmd_deg}


In [ ]:
# ================================================================================
# BƯỚC 6: HỢP NHẤT MA TRẬN MASTER BENCHMARK (CLEAN + DEGRADED)
# ================================================================================
import pandas as pd
from IPython.display import display

MASTER_CSV = f"{DRIVE_ROOT}/log/final_full_benchmark_matrix.csv"
LOCAL_REPO_LOG = "/content/fatformer-xla/DATASET/logs/m4_full_benchmark/final_full_benchmark_matrix.csv"

dfs = []
if os.path.exists(OUTPUT_CLEAN_CSV):
    clean_df = pd.read_csv(OUTPUT_CLEAN_CSV)
    clean_df["variant"] = "clean"
    dfs.append(clean_df)

if os.path.exists(OUTPUT_DEG_CSV):
    deg_df = pd.read_csv(OUTPUT_DEG_CSV)
    dfs.append(deg_df)

if dfs:
    master_df = pd.concat(dfs, ignore_index=True)
    master_df.to_csv(MASTER_CSV, index=False)
    if os.path.exists("/content/fatformer-xla/DATASET/logs/m4_full_benchmark"):
        master_df.to_csv(LOCAL_REPO_LOG, index=False)
    print(f"[✓] Đã tạo thành công Ma trận Master Benchmark: {MASTER_CSV} ({len(master_df)} hàng)!")
    display(master_df.head(25))
else:
    print("[!] Chưa tìm thấy kết quả đầu ra từ Bước 4 và Bước 5!")

In [ ]:
# ================================================================================
# BƯỚC 7: TRỰC QUAN HÓA TOÀN DIỆN: SO SÁNH FATFORMER-XLA VỚI BASELINE MỐC SÀN
# ================================================================================
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if os.path.exists(MASTER_CSV):
    df = pd.read_csv(MASTER_CSV)
    print("\n--- TỔNG KẾT THEO TỪNG BIẾN THỂ TRÊN 18 TẬP TEST ---")
    summary_rows = df[df["subset"] == "MEAN_OVERALL"]
    if summary_rows.empty:
        summary_rows = df.groupby("variant")[["acc", "f_acc", "r_acc"]].mean().reset_index()
    display(summary_rows)
    
    # Vẽ biểu đồ Master Comparison
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    fig, ax = plt.subplots(figsize=(12, 6), dpi=300)
    
    variants = summary_rows["variant"].tolist()
    acc_vals = summary_rows["acc"].tolist()
    fake_vals = summary_rows["f_acc"].tolist()
    
    x = np.arange(len(variants))
    width = 0.35
    
    r1 = ax.bar(x - width/2, acc_vals, width, label='Overall Accuracy (%)', color='#2ecc71', edgecolor='black')
    r2 = ax.bar(x + width/2, fake_vals, width, label='Fake Accuracy (%)', color='#3498db', edgecolor='black')
    
    ax.set_ylabel('Độ chính xác (%)', fontsize=12, fontweight='bold')
    ax.set_title('HIỆU NĂNG FATFORMER-XLA TRÊN 18 TẬP TEST HAI CHIỀU\n(Clean vs 6 Biến thể Suy biến Vật lý)', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(variants, fontsize=11, fontweight='bold')
    ax.set_ylim(0, 115)
    ax.legend(loc='upper right', frameon=True, fontsize=11)
    
    for rect in list(r1) + list(r2):
        h = rect.get_height()
        ax.annotate(f'{h:.1f}%', xy=(rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9, fontweight='bold')
                    
    chart_path = f"{DRIVE_ROOT}/log/task_5.1_master_benchmark_chart.png"
    plt.tight_layout()
    plt.savefig(chart_path, dpi=300)
    if os.path.exists("/content/fatformer-xla/DATASET/logs/m4_full_benchmark"):
        plt.savefig("/content/fatformer-xla/DATASET/logs/m4_full_benchmark/task_5.1_master_benchmark_chart.png", dpi=300)
    plt.show()
    print(f"[✓] Biểu đồ đã lưu thành công tại: {chart_path}")